In [ ]:
# Load all the environment variables from ~/.bashrc
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

paths = []
if os.path.join(os.path.expanduser("~"), ".bashrc"):
    paths.append(os.path.join(os.path.expanduser("~"), ".bashrc"))
if os.path.join(os.path.expanduser("~"), ".zshrc"):
    paths.append(os.path.join(os.path.expanduser("~"), ".zshrc"))

for path in paths:
    with open(path) as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith("export"):
                var = line.split()[1].split("=")[0]
                if var == "PATH":
                    continue
                os.environ[var] = line.split("=")[1].strip().strip('"')
                print(var, os.environ[var])

OPENSSL_ROOT_DIR /usr/local/opt/openssl@3
ZSH /Users/guillemcv/.oh-my-zsh
PL_API_KEY PLAKfda103372c864062b8900aaa4c9639bd
LUPNT_DATA_PATH /Users/guillemcv/Development/NavLab/LuPNT/data
CLOUDSDK_PYTHON $(which python)
CODECOV_TOKEN d0f7e7f7-f53c-4a81-8d30-2539383403f5
GMAT_PATH /Users/guillemcv/Applications/GMAT R2022a
PYTHONPATH ${PYTHONPATH}:/Users/guillemcv/Development/NavLab/LuPNT/


In [2]:
# import pylupnt as pnt
import numpy as np
import os
from utils.gmat import gmat, gmat_path
from utils import data, gmat_helpers

In [4]:
kwargs = {}
coe = data.coe_array_elfo
earth_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "earth", "JGM3.cof"
)
luna_potential_file_path = os.path.join(
    gmat_path, "data", "gravity", "luna", "grgm900c.cof"
)

EarthMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")

# Spacecraft configuration preliminaries
sc = gmat.Construct("Spacecraft", "LunaOrbiter")
sc.SetField("DateFormat", "UTCGregorian")
sc.SetField("Epoch", "20 Jul 2020 12:00:00.000")
sc.SetField("CoordinateSystem", "LunaMJ2000Eq")
# sc.SetField("DisplayStateType", "Keplerian")

# Orbital state
sc.SetField("SMA", coe[0])
sc.SetField("ECC", coe[1])
sc.SetField("INC", np.rad2deg(coe[2]))
sc.SetField("RAAN", np.rad2deg(coe[3]))
sc.SetField("AOP", np.rad2deg(coe[4]))
sc.SetField("TA", np.rad2deg())

# Spacecraft ballistic properties for the SRP and Drag models
if "SRPArea" in kwargs:
    sc.SetField("SRPArea", 2.5)
if "Cr" in kwargs:
    sc.SetField("Cr", 1.75)
if "DragArea" in kwargs:
    sc.SetField("DragArea", 1.8)
if "Cd" in kwargs:
    sc.SetField("Cd", 2.1)

sc.SetField("DryMass", 80)

# Force model settings
fm = gmat.Construct("ForceModel", "FM")
fm.SetField("CentralBody", "Luna")

# An 8x8 JGM-3 Gravity Model
grav = gmat.Construct("GravityField")
grav.SetField("BodyName", "Luna")
grav.SetField("PotentialFile", luna_potential_file_path)
grav.SetField("Degree", 0)
grav.SetField("Order", 0)

# Add forces into the ODEModel container
fm.AddForce(grav)

gmat.Initialize()

# Build the propagation container class
pdprop = gmat.Construct("Propagator", "PDProp")

# Create and assign a numerical integrator for use in the propagation
gator = gmat.Construct("PrinceDormand78", "Gator")
pdprop.SetReference(gator)

# Set some of the fields for the integration
pdprop.SetField("InitialStepSize", 60.0)
pdprop.SetField("Accuracy", 1.0e-12)
pdprop.SetField("MinStep", 0.0)

# Perform top level initialization
gmat.Initialize()

# Setup the spacecraft that is propagated
pdprop.AddPropObject(sc)
pdprop.PrepareInternals()

# Refresh the 'gator reference
fm = pdprop.GetODEModel()
gator = pdprop.GetPropagator()

# Take a 60 second step, showing the state before and after propagation
print(sc.GetEpoch())
print(gator.GetState())
print(sc.GetState().GetState())
print(sc.GetCartesianState())

29051.000428638676
[-159692.38629611005, 310994.93081171275, 150197.53605797305, -1.6742435746990314, -0.47811233991680757, 1.393762309062989]
[-159692.38629611005, 310994.93081171275, 150197.53605797305, -1.6742435746990314, -0.47811233991680757, 1.393762309062989]
0                2925.150000000023 0                -0.7531820164291154 5.551115123125783e-17 1.510649418797653


In [4]:
gator.GetState()

[-162550.00497932723,
 305071.7865004737,
 155929.03244103043,
 -1.0433590712049405,
 -1.3773987417408444,
 0.12840375330478881]

In [5]:
gator.Step(60.0)
gator.GetState()

[-162612.60438448974,
 304989.1385621929,
 155936.73461447857,
 -1.0432877595378618,
 -1.3775325346790437,
 0.12833535777918567]

In [6]:
EarthMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "EarthMJ2000Eq", "Earth", "MJ2000Eq")
LunaMJ2000Eq = gmat.Construct(
    "CoordinateSystem", "LunaMJ2000Eq", "Luna", "MJ2000Eq")
epoch = gmat.GmatTime(sc.GetEpoch())
state_earth = gmat.Rvector6(*gator.GetState())
state_luna = gmat.Rvector6()
gmat.Initialize()

In [7]:
converter = gmat.CoordinateConverter()
converter.Convert(epoch, state_earth, EarthMJ2000Eq, state_luna, LunaMJ2000Eq)
print(state_luna)

-2920.218088268448 -3080.642249462078 5739.198556519637 -0.122226201268088 -0.8994201947619611 0.2452224675139842


In [8]:
state_luna.GetSize()

6

In [9]:
cart = pnt.classical_to_cartesian(data.coe_array_elfo, pnt.MU_MOON)
epoch = pnt.SpiceInterface.string_to_tai("2001/04/06 12:00:00.000 UTC")
pnt.FrameConverter.convert(
    epoch, cart, pnt.CoordSystem.MI, pnt.CoordSystem.GCRF)

Loaded All SPICE Kernels


array([-3.65467245e+05,  1.65928319e+04,  4.77190199e+04, -2.38777562e-01,
       -1.89248259e+00, -1.55980647e-01])

Loaded Chebyshev coefficients for 14 planets.


In [10]:
time_sys_converter = gmat.TimeSystemConverter.Instance()
GMAT_CSPICE_TAI_OFFSET = time_sys_converter.ConvertGregorianToMjd("01 Jan 2000 12:00:00.000") * 86400.0
print(GMAT_CSPICE_TAI_OFFSET)

1861488000.0


In [11]:
time_sys_converter = gmat.TimeSystemConverter.Instance()
epoch_gmat = (
    time_sys_converter.Convert(
        # time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000"),
        time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000"),
        gmat.TimeSystemConverter.UTC,
        gmat.TimeSystemConverter.TAI,
    )
)
print(epoch_gmat)

29051.00042824074


In [12]:
epoch = pnt.SpiceInterface.string_to_tai("2020/07/20 12:00:00.000 UTC")
print(epoch)

648518437.0


In [13]:
gmat_helpers.convert_pylupnt_to_gmat_epoch(epoch)

29051.00042824074

In [14]:
epoch_gmat = time_sys_converter.ConvertGregorianToMjd("20 Jul 2020 12:00:00.000")
print(epoch_gmat)

29051.0


In [15]:
gmat_helpers.convert_gmat_to_pylupnt_epoch(epoch_gmat)

648518400.0

In [16]:
from utils import gmat_helpers

In [17]:
cart = pnt.classical_to_cartesian(data.coe_array_elfo, pnt.MU_MOON)
epoch = pnt.SpiceInterface.string_to_tai("2020/07/20 12:00:00.000")

# frametems = pnt.CoordSystem.__members__.values()
frametems = [pnt.ITRF, pnt.GCRF, pnt.ICRF, pnt.MI, pnt.EMR, pnt.ME, pnt.PA]

In [21]:
cartt_MI = np.zeros(6)
cart_from = pnt.FrameConverter.convert(
    epoch, cart, pnt.CoordSystem.GCRF, pnt.CoordSystem.ITRF
)
print("LuPNT")
print("\n".join(map(str, cart_from)))
cart_from_gmat = gmat_helpers.convert_coord(
    epoch, cart, pnt.CoordSystem.GCRF, pnt.CoordSystem.ITRF
)
print("\nGMAT")
print("\n".join(map(str, cart_from_gmat)))

print("\nError")
pnt.utils.print_aligned(cart_from - cart_from_gmat)

LuPNT
-1268.1280707197964
3950.9554601656255
5725.889220543232
-0.4439356069859093
0.6290357260188614
0.24505526382037654

GMAT
-1268.1374588956844
3950.9534177413075
5725.888550616152
-0.4439370672660746
0.629034602120141
0.24505524108451163

Error

[[0.009388175888034311  ]
 [0.002042424318005942  ]
 [0.0006699270797980716 ]
 [1.4602801653440523e-06]
 [1.1238987204231776e-06]
 [2.2735864912970527e-08]



In [22]:
cart = np.zeros(6)
cart_from = pnt.FrameConverter.convert(
    epoch, cart, pnt.CoordSystem.GCRF, pnt.CoordSystem.MI
)
print("LuPNT")
print("\n".join(map(str, cart_from)))
cart_from_gmat = gmat_helpers.convert_coord(
    epoch, cart, pnt.CoordSystem.GCRF, pnt.CoordSystem.MI
)
print("\nGMAT")
print("\n".join(map(str, cart_from_gmat)))

# >> cspice_spkezr('399', epoch_tdb, 'J2000', 'NONE', '301')
matlab = np.array(
    [
        159692.398362043,
        -308069.775920662,
        -150197.533041719,
        0.92106154514156,
        0.47811236919014,
        0.116887125768653,
    ]
)

print("\nError")
pnt.utils.print_aligned(cart_from - cart_from_gmat)
pnt.utils.print_aligned(cart_from - matlab)
pnt.utils.print_aligned(cart_from_gmat - matlab)

LuPNT
159692.39852752635
-308069.7758805594
-150197.5330526558
0.921061545053458
0.47811236960258763
0.11688712595974197

GMAT
159692.35462853205
-308069.79724998237
-150197.54007673915
0.9210615987690121
0.4781122616498614
0.11688707151133704

Error

[[ 0.04389899430680089   ]
 [ 0.02136942296056077   ]
 [ 0.00702408334473148   ]
 [-5.371555400479622e-08 ]
 [ 1.0795272620267582e-07]
 [ 5.4448404937512684e-08]


[[ 0.00016548336134292185]
 [ 4.0102575439959764e-05]
 [-1.0936812032014132e-05]
 [-8.810197016373422e-11 ]
 [ 4.1244763160364073e-10]
 [ 1.9108897797437407e-10]


[[-0.043733510945457965  ]
 [-0.02132932038512081   ]
 [-0.007035020156763494  ]
 [ 5.3627452034632483e-08]
 [-1.0754027857107218e-07]
 [-5.425731595953831e-08 ]



In [23]:

def format_element(x, fmt="{}"):
    if x == 0.0:
        return "0.0"
    else:
        return fmt.format(x)


def print_aligned(matrix):
    if len(matrix.shape) == 1:
        num_columns = 1
        matrix = matrix.reshape((len(matrix), 1))
    else:
        num_columns = len(matrix[0])
    max_before_decimal = [0] * num_columns
    max_after_decimal = [0] * num_columns

    # Step 1: Find the maximum length before and after decimal for each column
    for row in matrix:
        for i, value in enumerate(row):
            parts = str(value).split(".")
            max_before_decimal[i] = max(max_before_decimal[i], len(parts[0]))
            if len(parts) > 1:
                max_after_decimal[i] = max(max_after_decimal[i], len(parts[1]))

    # Step 2: Format each number with padding and accumulate in a string
    txt = "\n["
    for r, row in enumerate(matrix):
        line = "[" if r == 0 else " ["
        for i, value in enumerate(row):
            value = float(format_element(value))
            before, after = (
                str(value).split(".") if "." in str(value) else (str(value), "")
            )
            padding_left = " " * (max_before_decimal[i] - len(before))
            padding_right = " " * (max_after_decimal[i] - len(after))
            formatted_value = f"{padding_left}{before}.{after}{padding_right}"
            line += formatted_value + (" " if i < num_columns - 1 else "")
        txt += line + "]\n"

    print(txt)
